# Tracking Political Change with Embeddings of Parliamentary Speeches
***
# Zero-Shot Topic Label Embeddings
## 1. Setup
### 1.1 Imports and Seeds

In [2]:
# Imports
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from embedding_utils import mean_pooling  # shared pooling logic, kept in sync with 02-01/02-02

In [3]:
SEED = 24
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


### 1.2 Backbone Path

In [4]:
# Path to the fine-tuned backbone saved by 02-02_finetuning_embedding.ipynb
FT_BACKBONE_PATH = "jina_v3_contrastive_backbone"

### 1.3 Embedding Helper

In [5]:
# Loads a model, embeds a list of texts, then frees GPU memory (mirrors 02-02's embed_with_model)
def embed_with_model(model_path, texts, batch_size=16, max_length=64, device=device):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModel.from_pretrained(model_path).to(device)
    mdl.eval()

    all_emb = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tok(batch, padding=True, truncation=True,
                         max_length=max_length, return_tensors="pt").to(device)
            out = mdl(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            pooled = mean_pooling(out.last_hidden_state, inputs["attention_mask"])
            emb = F.normalize(pooled, p=2, dim=1)
            all_emb.append(emb.cpu())

    del mdl     # frees this model in case another gets loaded later in the same session
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return torch.cat(all_emb, dim=0).numpy()

## 2. Topic Embeddings
### 2.1 Zero-Shot Topic Labels and Embedding

In [6]:
# Fixed policy topics for zero-shot assignment; max_length=64 easily covers these short labels
zeroshot_topic_list = [
    "Außenpolitik und internationale Beziehungen",
    "Innere Sicherheit und Polizei",
    "Migration, Asyl und Einwanderung",
    "Sport und Ehrenamt",
    "Recht und Verbraucherschutz",
    "Steuern und Finanzpolitik",
    "Bundeshaushalt und öffentliche Ausgaben",
    "Wirtschaftspolitik und Energieversorgung",
    "Landwirtschaft und Ernährung",
    "Arbeitsmarkt und Sozialpolitik",
    "Verteidigungspolitik und Bundeswehr",
    "Bildung, Familie und Jugend",
    "Gesundheitspolitik und Krankenversicherung",
    "Verkehr und Infrastruktur",
    "Umweltschutz, Klimapolitik und Atomkraft",
    "Menschenrechte und humanitäre Hilfe",
    "Forschung, Technologie und Raumfahrt",
    "Entwicklungszusammenarbeit",
    "Europapolitik und EU-Angelegenheiten",
    "Kultur und Medien",
    "Wohnungspolitik und Städtebau",
]

topic_embeddings = embed_with_model(
    FT_BACKBONE_PATH,
    zeroshot_topic_list,
    batch_size=16,
    max_length=64,
    device=device,
)

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

## 3. Save

In [7]:
print("Shape:", topic_embeddings.shape)  # (21, 1024)

topic_df = pd.DataFrame({
    "topic": zeroshot_topic_list,
    "embedding": list(topic_embeddings),
})
topic_df.to_parquet("zeroshot_topic_embeddings.parquet", engine="pyarrow", index=False)

Shape: (21, 1024)
